# ------------------------------ Data cleaning & validation ------------------------------

### Importing important packages

In [27]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as dt
import math as math
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

import pyodbc

# Connecting to sql server to import data

In [28]:
# Finding ODBC Driver

print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [29]:
# ============================================================
# 1. SQL Server connection
# ============================================================

server = r"DESKTOP-0SIEUH6\SQLEXPRESS"
database = "EuroEatsDB"

conn = pyodbc.connect(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# ============================================================
# 2. Get all tables from dbo schema
# ============================================================

tables_query = """
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'dbo'
  AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME;
"""

tables = pd.read_sql(tables_query, conn)

print("Tables found:")
print(tables)

Tables found:
    TABLE_NAME
0    customers
1   deliveries
2      drivers
3   menu_items
4  order_items
5       orders
6     payments
7  restaurants
8      reviews


### Importing datasets from sql server in form of tables

In [30]:
# Dictionary to hold all DataFrames
dataframes = {}

for table_name in tables["TABLE_NAME"]:
    
    query = f"SELECT * FROM dbo.[{table_name}]"
    
    dataframes[table_name] = pd.read_sql(query, conn)
    
    # Create a variable with the same name as the SQL table
    globals()[table_name] = dataframes[table_name]
    
    print(f"{table_name}: {dataframes[table_name].shape}")

customers: (1500, 10)
deliveries: (1000, 12)
drivers: (350, 11)
menu_items: (1500, 9)
order_items: (4697, 6)
orders: (4500, 11)
payments: (800, 7)
restaurants: (120, 11)
reviews: (500, 8)


In [31]:
# ============================================================
# BLOCK 1 — CREATE WORKING COPIES
# ============================================================

import pandas as pd
import numpy as np

customers = customers.copy()
restaurants = restaurants.copy()
menu_items = menu_items.copy()
drivers = drivers.copy()
orders = orders.copy()
order_items = order_items.copy()
deliveries = deliveries.copy()
payments = payments.copy()
reviews = reviews.copy()

print("Working copies created successfully.")

Working copies created successfully.


In [32]:
# ============================================================
# BLOCK 2 — DATASET STRUCTURE VALIDATION
# ============================================================

dataframes = {
    "customers": customers,
    "restaurants": restaurants,
    "menu_items": menu_items,
    "drivers": drivers,
    "orders": orders,
    "order_items": order_items,
    "deliveries": deliveries,
    "payments": payments,
    "reviews": reviews
}

for name, df in dataframes.items():
    print(f"\n{name.upper()}")
    print("-" * 50)
    print("Rows    :", df.shape[0])
    print("Columns :", df.shape[1])
    print("Columns :", list(df.columns))


CUSTOMERS
--------------------------------------------------
Rows    : 1500
Columns : 10
Columns : ['customer_id', 'first_name', 'last_name', 'age', 'gender', 'country', 'city', 'registration_date', 'customer_segment', 'loyalty_status']

RESTAURANTS
--------------------------------------------------
Rows    : 120
Columns : 11
Columns : ['restaurant_id', 'restaurant_name', 'country', 'city', 'cuisine_type', 'price_level', 'restaurant_rating', 'average_preparation_time', 'restaurant_join_date', 'restaurant_size', 'average_daily_orders']

MENU_ITEMS
--------------------------------------------------
Rows    : 1500
Columns : 9
Columns : ['item_id', 'restaurant_id', 'item_name', 'category', 'price_eur', 'preparation_time_minutes', 'calories', 'vegetarian', 'popularity_score']

DRIVERS
--------------------------------------------------
Rows    : 350
Columns : 11
Columns : ['driver_id', 'first_name', 'last_name', 'country', 'city', 'vehicle_type', 'years_experience', 'driver_rating', 'comple

In [33]:
# ============================================================
# BLOCK 3 — DUPLICATE ROW VALIDATION
# ============================================================

for name, df in dataframes.items():
    
    duplicate_count = df.duplicated().sum()
    
    print(f"{name:15} | Duplicate rows: {duplicate_count}")

customers       | Duplicate rows: 0
restaurants     | Duplicate rows: 0
menu_items      | Duplicate rows: 0
drivers         | Duplicate rows: 0
orders          | Duplicate rows: 0
order_items     | Duplicate rows: 0
deliveries      | Duplicate rows: 0
payments        | Duplicate rows: 0
reviews         | Duplicate rows: 0


In [34]:
# ============================================================
# BLOCK 4 — PRIMARY KEY DUPLICATE VALIDATION
# ============================================================

primary_keys = {
    "customers": "customer_id",
    "restaurants": "restaurant_id",
    "menu_items": "item_id",
    "drivers": "driver_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "deliveries": "delivery_id",
    "payments": "payment_id",
    "reviews": "review_id"
}

for table, pk in primary_keys.items():
    
    df = dataframes[table]
    
    duplicate_pk = df[pk].duplicated().sum()
    
    print(f"{table:15} | Duplicate {pk}: {duplicate_pk}")

customers       | Duplicate customer_id: 0
restaurants     | Duplicate restaurant_id: 0
menu_items      | Duplicate item_id: 0
drivers         | Duplicate driver_id: 0
orders          | Duplicate order_id: 0
order_items     | Duplicate order_item_id: 0
deliveries      | Duplicate delivery_id: 0
payments        | Duplicate payment_id: 0
reviews         | Duplicate review_id: 0


In [35]:
# ============================================================
# BLOCK 4 — PRIMARY KEY DUPLICATE VALIDATION
# ============================================================

primary_keys = {
    "customers": "customer_id",
    "restaurants": "restaurant_id",
    "menu_items": "item_id",
    "drivers": "driver_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "deliveries": "delivery_id",
    "payments": "payment_id",
    "reviews": "review_id"
}

for table, pk in primary_keys.items():
    
    df = dataframes[table]
    
    duplicate_pk = df[pk].duplicated().sum()
    
    print(f"{table:15} | Duplicate {pk}: {duplicate_pk}")

customers       | Duplicate customer_id: 0
restaurants     | Duplicate restaurant_id: 0
menu_items      | Duplicate item_id: 0
drivers         | Duplicate driver_id: 0
orders          | Duplicate order_id: 0
order_items     | Duplicate order_item_id: 0
deliveries      | Duplicate delivery_id: 0
payments        | Duplicate payment_id: 0
reviews         | Duplicate review_id: 0


In [36]:
# ============================================================
# BLOCK 5 — NULL VALUE ANALYSIS
# ============================================================

for name, df in dataframes.items():
    
    print(f"\n{name.upper()}")
    print("-" * 50)
    
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    
    if len(nulls) == 0:
        print("No missing values.")
    else:
        print(nulls)


CUSTOMERS
--------------------------------------------------
No missing values.

RESTAURANTS
--------------------------------------------------
No missing values.

MENU_ITEMS
--------------------------------------------------
No missing values.

DRIVERS
--------------------------------------------------
No missing values.

ORDERS
--------------------------------------------------
No missing values.

ORDER_ITEMS
--------------------------------------------------
No missing values.

DELIVERIES
--------------------------------------------------
No missing values.

PAYMENTS
--------------------------------------------------
No missing values.

REVIEWS
--------------------------------------------------
No missing values.


In [37]:
# ============================================================
# BLOCK 6 — TEXT STANDARDIZATION
# ============================================================

for name, df in dataframes.items():
    
    text_columns = df.select_dtypes(include="object").columns
    
    for col in text_columns:
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )

print("Text columns standardized.")

Text columns standardized.


In [38]:
# ============================================================
# BLOCK 7 — NUMERIC DATATYPE CONVERSION
# ============================================================

numeric_columns = {
    "customers": [
        "age"
    ],

    "restaurants": [
        "price_level",
        "restaurant_rating",
        "average_preparation_time",
        "average_daily_orders"
    ],

    "menu_items": [
        "price_eur",
        "preparation_time_minutes",
        "calories",
        "popularity_score"
    ],

    "drivers": [
        "years_experience",
        "driver_rating",
        "completed_deliveries",
        "average_speed_kmh"
    ],

    "orders": [
        "number_of_items",
        "subtotal_eur",
        "delivery_fee_eur",
        "discount_eur",
        "total_amount_eur"
    ],

    "order_items": [
        "quantity",
        "unit_price_eur",
        "item_discount_eur"
    ],

    "deliveries": [
        "distance_km",
        "temperature_c",
        "pickup_delay_minutes",
        "restaurant_wait_minutes",
        "estimated_delivery_minutes",
        "actual_delivery_minutes"
    ],

    "payments": [
        "amount_eur"
    ],

    "reviews": [
        "rating",
        "delivery_rating",
        "food_rating"
    ]
}

for table, columns in numeric_columns.items():
    
    df = dataframes[table]
    
    for col in columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

print("Numeric datatype conversion completed.")

Numeric datatype conversion completed.


In [39]:
# ============================================================
# BLOCK 7 — NUMERIC DATATYPE CONVERSION
# ============================================================

numeric_columns = {
    "customers": [
        "age"
    ],

    "restaurants": [
        "price_level",
        "restaurant_rating",
        "average_preparation_time",
        "average_daily_orders"
    ],

    "menu_items": [
        "price_eur",
        "preparation_time_minutes",
        "calories",
        "popularity_score"
    ],

    "drivers": [
        "years_experience",
        "driver_rating",
        "completed_deliveries",
        "average_speed_kmh"
    ],

    "orders": [
        "number_of_items",
        "subtotal_eur",
        "delivery_fee_eur",
        "discount_eur",
        "total_amount_eur"
    ],

    "order_items": [
        "quantity",
        "unit_price_eur",
        "item_discount_eur"
    ],

    "deliveries": [
        "distance_km",
        "temperature_c",
        "pickup_delay_minutes",
        "restaurant_wait_minutes",
        "estimated_delivery_minutes",
        "actual_delivery_minutes"
    ],

    "payments": [
        "amount_eur"
    ],

    "reviews": [
        "rating",
        "delivery_rating",
        "food_rating"
    ]
}

for table, columns in numeric_columns.items():
    
    df = dataframes[table]
    
    for col in columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

print("Numeric datatype conversion completed.")

Numeric datatype conversion completed.


In [40]:
# ============================================================
# BLOCK 8 — DATE DATATYPE CONVERSION
# ============================================================

date_columns = {
    "customers": [
        "registration_date"
    ],

    "restaurants": [
        "restaurant_join_date"
    ],

    "orders": [
        "order_datetime"
    ],

    "payments": [
        "payment_datetime"
    ],

    "reviews": [
        "review_datetime"
    ]
}

for table, columns in date_columns.items():
    
    df = dataframes[table]
    
    for col in columns:
        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

print("Date conversion completed.")

Date conversion completed.


In [41]:
# ============================================================
# BLOCK 9 — INVALID VALUE VALIDATION
# ============================================================

print("Invalid numeric/date values after conversion")
print("=" * 60)

for table, columns in {**numeric_columns, **date_columns}.items():
    
    df = dataframes[table]
    
    for col in columns:
        
        invalid_count = df[col].isna().sum()
        
        if invalid_count > 0:
            print(
                f"{table:15} | "
                f"{col:35} | "
                f"Invalid/NULL: {invalid_count}"
            )

Invalid numeric/date values after conversion


In [42]:
# ============================================================
# BLOCK 10 — NUMERIC RANGE VALIDATION
# ============================================================

range_checks = {
    "customers": {
        "age": (0, 120)
    },

    "restaurants": {
        "restaurant_rating": (0, 5),
        "average_preparation_time": (0, None),
        "average_daily_orders": (0, None)
    },

    "menu_items": {
        "price_eur": (0, None),
        "preparation_time_minutes": (0, None),
        "calories": (0, None),
        "popularity_score": (0, 100)
    },

    "drivers": {
        "years_experience": (0, None),
        "driver_rating": (0, 5),
        "completed_deliveries": (0, None),
        "average_speed_kmh": (0, None)
    },

    "orders": {
        "number_of_items": (1, None),
        "subtotal_eur": (0, None),
        "delivery_fee_eur": (0, None),
        "discount_eur": (0, None),
        "total_amount_eur": (0, None)
    },

    "order_items": {
        "quantity": (1, None),
        "unit_price_eur": (0, None),
        "item_discount_eur": (0, None)
    },

    "deliveries": {
        "distance_km": (0, None),
        "pickup_delay_minutes": (0, None),
        "restaurant_wait_minutes": (0, None),
        "estimated_delivery_minutes": (0, None),
        "actual_delivery_minutes": (0, None)
    },

    "reviews": {
        "rating": (1, 5),
        "delivery_rating": (1, 5),
        "food_rating": (1, 5)
    }
}

for table, columns in range_checks.items():
    
    df = dataframes[table]
    
    for col, (minimum, maximum) in columns.items():
        
        if minimum is not None:
            count = (df[col] < minimum).sum()
            
            if count > 0:
                print(
                    f"{table}.{col}: "
                    f"{count} values below {minimum}"
                )
        
        if maximum is not None:
            count = (df[col] > maximum).sum()
            
            if count > 0:
                print(
                    f"{table}.{col}: "
                    f"{count} values above {maximum}"
                )

print("Range validation completed.")

Range validation completed.


In [43]:
# ============================================================
# BLOCK 11 — CATEGORICAL COLUMNS
# ============================================================

categorical_columns = {
    "customers": [
        "gender",
        "country",
        "city",
        "customer_segment",
        "loyalty_status"
    ],

    "restaurants": [
        "country",
        "city",
        "cuisine_type",
        "price_level",
        "restaurant_size"
    ],

    "menu_items": [
        "category",
        "vegetarian"
    ],

    "drivers": [
        "country",
        "city",
        "vehicle_type",
        "employment_type"
    ],

    "orders": [
        "payment_method",
        "order_status"
    ],

    "deliveries": [
        "traffic_level",
        "weather_condition",
        "delivery_status"
    ],

    "payments": [
        "payment_method",
        "payment_status",
        "currency"
    ],

    "reviews": [
        "review_sentiment"
    ]
}

In [44]:
# ============================================================
# BLOCK 12 — CATEGORICAL TEXT STANDARDIZATION
# ============================================================

for table, columns in categorical_columns.items():
    
    df = dataframes[table]
    
    for col in columns:
        
        df[col] = df[col].apply(
            lambda x: x.title() if isinstance(x, str) else x
        )

print("Categorical text standardized.")

Categorical text standardized.


In [45]:
# ============================================================
# BLOCK 13 — REFERENTIAL INTEGRITY VALIDATION
# ============================================================

foreign_keys = [
    ("menu_items", "restaurant_id", "restaurants", "restaurant_id"),
    
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "restaurant_id", "restaurants", "restaurant_id"),
    
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "item_id", "menu_items", "item_id"),
    
    ("deliveries", "order_id", "orders", "order_id"),
    ("deliveries", "driver_id", "drivers", "driver_id"),
    
    ("payments", "order_id", "orders", "order_id"),
    
    ("reviews", "order_id", "orders", "order_id"),
    ("reviews", "customer_id", "customers", "customer_id")
]

for child_table, child_col, parent_table, parent_col in foreign_keys:
    
    child_values = set(
        dataframes[child_table][child_col].dropna()
    )
    
    parent_values = set(
        dataframes[parent_table][parent_col].dropna()
    )
    
    missing_keys = child_values - parent_values
    
    print(
        f"{child_table}.{child_col} -> "
        f"{parent_table}.{parent_col} | "
        f"Missing keys: {len(missing_keys)}"
    )

menu_items.restaurant_id -> restaurants.restaurant_id | Missing keys: 0
orders.customer_id -> customers.customer_id | Missing keys: 0
orders.restaurant_id -> restaurants.restaurant_id | Missing keys: 0
order_items.order_id -> orders.order_id | Missing keys: 0
order_items.item_id -> menu_items.item_id | Missing keys: 0
deliveries.order_id -> orders.order_id | Missing keys: 0
deliveries.driver_id -> drivers.driver_id | Missing keys: 0
payments.order_id -> orders.order_id | Missing keys: 0
reviews.order_id -> orders.order_id | Missing keys: 0
reviews.customer_id -> customers.customer_id | Missing keys: 0


In [46]:
# ============================================================
# BLOCK 14 — BUSINESS LOGIC VALIDATION
# ============================================================

# ------------------------------------------------------------
# 1. Order total consistency
# ------------------------------------------------------------

orders["calculated_total"] = (
    orders["subtotal_eur"]
    + orders["delivery_fee_eur"]
    - orders["discount_eur"]
)

orders["total_difference"] = (
    orders["total_amount_eur"]
    - orders["calculated_total"]
)

print(
    "Orders with total amount mismatch:",
    (
        orders["total_difference"].abs() > 0.01
    ).sum()
)


# ------------------------------------------------------------
# 2. Delivery time should be greater than zero
# ------------------------------------------------------------

invalid_delivery_time = (
    deliveries["actual_delivery_minutes"].notna()
    &
    (deliveries["actual_delivery_minutes"] <= 0)
).sum()

print(
    "Invalid actual delivery times:",
    invalid_delivery_time
)


# ------------------------------------------------------------
# 3. Estimated delivery time should be greater than zero
# ------------------------------------------------------------

invalid_estimated_time = (
    deliveries["estimated_delivery_minutes"].notna()
    &
    (deliveries["estimated_delivery_minutes"] <= 0)
).sum()

print(
    "Invalid estimated delivery times:",
    invalid_estimated_time
)


# ------------------------------------------------------------
# 4. Discount should not exceed order subtotal
# ------------------------------------------------------------

invalid_discount = (
    orders["discount_eur"]
    > orders["subtotal_eur"]
).sum()

print(
    "Discount greater than subtotal:",
    invalid_discount
)

Orders with total amount mismatch: 0
Invalid actual delivery times: 0
Invalid estimated delivery times: 0
Discount greater than subtotal: 0


In [47]:
# ============================================================
# BLOCK 14 — BUSINESS LOGIC VALIDATION
# ============================================================

# ------------------------------------------------------------
# 1. Order total consistency
# ------------------------------------------------------------

orders["calculated_total"] = (
    orders["subtotal_eur"]
    + orders["delivery_fee_eur"]
    - orders["discount_eur"]
)

orders["total_difference"] = (
    orders["total_amount_eur"]
    - orders["calculated_total"]
)

print(
    "Orders with total amount mismatch:",
    (
        orders["total_difference"].abs() > 0.01
    ).sum()
)


# ------------------------------------------------------------
# 2. Delivery time should be greater than zero
# ------------------------------------------------------------

invalid_delivery_time = (
    deliveries["actual_delivery_minutes"].notna()
    &
    (deliveries["actual_delivery_minutes"] <= 0)
).sum()

print(
    "Invalid actual delivery times:",
    invalid_delivery_time
)


# ------------------------------------------------------------
# 3. Estimated delivery time should be greater than zero
# ------------------------------------------------------------

invalid_estimated_time = (
    deliveries["estimated_delivery_minutes"].notna()
    &
    (deliveries["estimated_delivery_minutes"] <= 0)
).sum()

print(
    "Invalid estimated delivery times:",
    invalid_estimated_time
)


# ------------------------------------------------------------
# 4. Discount should not exceed order subtotal
# ------------------------------------------------------------

invalid_discount = (
    orders["discount_eur"]
    > orders["subtotal_eur"]
).sum()

print(
    "Discount greater than subtotal:",
    invalid_discount
)

Orders with total amount mismatch: 0
Invalid actual delivery times: 0
Invalid estimated delivery times: 0
Discount greater than subtotal: 0


In [48]:
# ============================================================
# BLOCK 16 — CATEGORICAL CONSISTENCY VALIDATION
# ============================================================

print("CATEGORICAL CONSISTENCY CHECK")
print("=" * 70)

for table, columns in categorical_columns.items():

    df = dataframes[table]

    print(f"\n{table.upper()}")
    print("-" * 50)

    for col in columns:

        values = (
            df[col]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
            .value_counts()
        )

        print(f"\n{col}:")
        print(values)

CATEGORICAL CONSISTENCY CHECK

CUSTOMERS
--------------------------------------------------

gender:
gender
female    753
male      696
other      51
Name: count, dtype: int64

country:
country
germany        256
italy          183
france         182
spain          137
poland         111
netherlands    101
sweden          84
belgium         79
greece          62
czechia         59
portugal        57
austria         53
norway          48
finland         44
denmark         44
Name: count, dtype: int64

city:
city
berlin          108
paris            89
rome             87
warsaw           61
madrid           61
brussels         60
amsterdam        60
munich           57
stockholm        55
athens           45
milan            42
prague           40
lisbon           40
vienna           37
marseille        37
hamburg          36
lyon             33
barcelona        33
oslo             32
krakow           32
naples           30
helsinki         30
cologne          30
gothenburg       29
cop

In [49]:
# ============================================================
# BLOCK 16 — CATEGORICAL CONSISTENCY VALIDATION
# ============================================================

print("CATEGORICAL CONSISTENCY CHECK")
print("=" * 70)

for table, columns in categorical_columns.items():

    df = dataframes[table]

    print(f"\n{table.upper()}")
    print("-" * 50)

    for col in columns:

        values = (
            df[col]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
            .value_counts()
        )

        print(f"\n{col}:")
        print(values)

CATEGORICAL CONSISTENCY CHECK

CUSTOMERS
--------------------------------------------------

gender:
gender
female    753
male      696
other      51
Name: count, dtype: int64

country:
country
germany        256
italy          183
france         182
spain          137
poland         111
netherlands    101
sweden          84
belgium         79
greece          62
czechia         59
portugal        57
austria         53
norway          48
finland         44
denmark         44
Name: count, dtype: int64

city:
city
berlin          108
paris            89
rome             87
warsaw           61
madrid           61
brussels         60
amsterdam        60
munich           57
stockholm        55
athens           45
milan            42
prague           40
lisbon           40
vienna           37
marseille        37
hamburg          36
lyon             33
barcelona        33
oslo             32
krakow           32
naples           30
helsinki         30
cologne          30
gothenburg       29
cop

In [50]:
# ============================================================
# BLOCK 18 — CROSS-TABLE BUSINESS CONSISTENCY
# ============================================================

print("CROSS-TABLE BUSINESS CONSISTENCY CHECK")
print("=" * 70)


# ------------------------------------------------------------
# 1. Review customer should match order customer
# ------------------------------------------------------------

review_customer_check = reviews[
    ["order_id", "customer_id"]
].merge(
    orders[
        ["order_id", "customer_id"]
    ],
    on="order_id",
    how="left",
    suffixes=("_review", "_order")
)

invalid_review_customers = (
    review_customer_check["customer_id_review"]
    != review_customer_check["customer_id_order"]
).sum()

print(
    "Reviews with customer mismatch:",
    invalid_review_customers
)


# ------------------------------------------------------------
# 2. Order item restaurant should match order restaurant
# ------------------------------------------------------------

order_item_check = (
    order_items[
        ["order_id", "item_id"]
    ]
    .merge(
        orders[
            ["order_id", "restaurant_id"]
        ],
        on="order_id",
        how="left"
    )
    .merge(
        menu_items[
            ["item_id", "restaurant_id"]
        ],
        on="item_id",
        how="left",
        suffixes=("_order", "_item")
    )
)

invalid_restaurant_items = (
    order_item_check["restaurant_id_order"]
    != order_item_check["restaurant_id_item"]
).sum()

print(
    "Order items from wrong restaurant:",
    invalid_restaurant_items
)


# ------------------------------------------------------------
# 3. Payment amount vs order amount
# ------------------------------------------------------------

payment_order_check = payments[
    ["order_id", "amount_eur"]
].merge(
    orders[
        ["order_id", "total_amount_eur"]
    ],
    on="order_id",
    how="left"
)

payment_difference = (
    payment_order_check["amount_eur"]
    - payment_order_check["total_amount_eur"]
).abs()

invalid_payment_amounts = (
    payment_difference > 0.01
).sum()

print(
    "Payments with amount mismatch:",
    invalid_payment_amounts
)


# ------------------------------------------------------------
# 4. Delivery should belong to an existing order
# ------------------------------------------------------------

delivery_order_check = deliveries[
    ["delivery_id", "order_id"]
].merge(
    orders[
        ["order_id"]
    ],
    on="order_id",
    how="left",
    indicator=True
)

missing_delivery_orders = (
    delivery_order_check["_merge"] == "left_only"
).sum()

print(
    "Deliveries without matching orders:",
    missing_delivery_orders
)


# ------------------------------------------------------------
# 5. Payment should belong to an existing order
# ------------------------------------------------------------

payment_order_exists = payments[
    ["payment_id", "order_id"]
].merge(
    orders[
        ["order_id"]
    ],
    on="order_id",
    how="left",
    indicator=True
)

missing_payment_orders = (
    payment_order_exists["_merge"] == "left_only"
).sum()

print(
    "Payments without matching orders:",
    missing_payment_orders
)

CROSS-TABLE BUSINESS CONSISTENCY CHECK
Reviews with customer mismatch: 0
Order items from wrong restaurant: 0
Payments with amount mismatch: 0
Deliveries without matching orders: 0
Payments without matching orders: 0


In [51]:
# ============================================================
# SAVE CLEANED DATA
# ============================================================

cleaned_data = {
    "customers": customers,
    "restaurants": restaurants,
    "menu_items": menu_items,
    "drivers": drivers,
    "orders": orders,
    "order_items": order_items,
    "deliveries": deliveries,
    "payments": payments,
    "reviews": reviews
}

pd.to_pickle(
    cleaned_data,
    "EuroEats_cleaned_data.pkl"
)

print("Cleaned EuroEats data saved successfully.")

Cleaned EuroEats data saved successfully.


###### ------------------------------------------------------------------------------------------------------------ THE END ------------------------------------------------------------------------------------------------------------